# Item2Vec cho Outfit Pairing — Colab Notebook (A100 40GB)

Notebook tự chứa để train Item2Vec (SGNS + hard negatives) cho project mRAG fashion. Output là một kNN graph CSV cùng schema với `final_outfit_graph.csv` để plug-in vào eval harness ở local.

**Tóm tắt kỹ thuật**: Item2Vec (Barkan & Koenigstein, 2016) là biến thể của word2vec áp dụng cho recsys. Mỗi basket được coi là một 'sentence', mỗi item là một 'word'. Train skip-gram với negative sampling (SGNS). Khác word2vec ở chỗ basket không có thứ tự nên context window = toàn bộ basket. **Không phải GNN** → không bị oversmoothing như LightGCN khi inject content features.

## 1. Input / Output specification

### Input bắt buộc (upload lên Drive trước khi chạy)

Upload 2 files vào `/content/drive/MyDrive/mRAG/`:

- `eval_baskets.pkl` (hoặc bất kỳ tên `eval_baskets*.pkl` — notebook auto-detect) — file pickle ~100MB, copy từ `data/processed/eval_cache/eval_baskets_<hash>.pkl` ở repo local. Bên trong là dict `{"train_baskets", "test_baskets", "cutoff_date", "valid_ids"}` với `train_baskets: List[List[str]]` (mỗi basket là list `article_id` 10-digit zero-padded).
- `dataset_qwen_completed.csv` — meta file (file final của ông), dùng để map item → product_type cho hard negative sampling. Chỉ cần 2 cột: `article_id`, `product_type_name`.

### Output (notebook sẽ tạo trong `/content/drive/MyDrive/mRAG/item2vec_output/`)

- `item2vec_{tag}_k20.csv` — kNN graph CSV. Schema: `item_a, item_b, weight`. weight là cosine similarity scaled int [0, 1000]. Bidirectional. **Đây là artifact chính** để mang về local + chạy eval.
- `item2vec_{tag}_embeddings.npz` — `embeddings` (shape `[vocab_size, embed_dim]`, L2-normalized) và `items` (list article_id tương ứng).
- `item2vec_{tag}_metadata.json` — config + loss history + stats.

Tag format: `d{embed_dim}_n{num_neg}_h{hard_ratio*100}_e{epochs}`. Ví dụ: `d256_n10_h60_e10`.

## 2. Config tối ưu cho A100 40GB

| Hyperparam | CPU baseline (local) | A100 40GB (notebook này) | Lý do |
|---|---|---|---|
| `embed_dim` | 128 | **256** | A100 thừa VRAM cho model lớn hơn → capacity cao hơn cho item interaction patterns. |
| `batch_size` | 8192 | **32768** | A100 throughput cực cao; batch lớn giảm Python overhead. Có thể push 65536 nếu muốn. |
| `num_negatives` | 5 | **10** | Nhiều negative samples → contrast signal mạnh hơn. A100 dư memory. |
| `hard_negative_ratio` | 0.6 | **0.6** | Giữ nguyên — đây là core của việc học cross-PT pairing. |
| `epochs` | 5 | **10** | Loss vẫn giảm sau 5 epochs trên CPU run → train lâu hơn với A100 dễ. |
| `precision` | fp32 | **bf16 autocast** | A100 native bf16 — tăng tốc ~2x mà không loss accuracy. |
| `lr` | 5e-3 | 5e-3 | Adam đủ robust với batch lớn ở scale này. |
| `knn_chunk` | 1024 | **4096** | Larger chunk cho kNN compute nhanh hơn. |

**Ước tính VRAM** (cấu hình A100 ở trên):
- Model + Adam state: `~330MB` (vocab 53K * 256 * 2 in/out * 4 bytes + Adam m/v)
- Forward activations: `~180MB` (batch * (1+num_neg) * embed_dim * 2 bytes bf16)
- Backward + gradients: `~500MB`
- **Tổng**: ~1GB. Còn thừa 39GB → có thể tăng tiếp batch/dim nếu muốn.

**Ước tính thời gian**:
- Data load (Path A từ pkl): 30s
- Materialize pairs: 60s (~58M pairs)
- 1 epoch trên A100: 20-40s
- 10 epochs + kNN export: ~5-10 phút tổng
- So sánh: CPU local 1 epoch = 12 phút → A100 nhanh hơn ~20x.

## 3. Setup môi trường

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("VRAM total:", f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB" if torch.cuda.is_available() else "n/a")
print("bf16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

assert torch.cuda.is_available(), "Chọn Runtime > Change runtime type > GPU (A100)"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/mRAG")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Drive root:", DRIVE_ROOT)
print("Files có sẵn:", os.listdir(DRIVE_ROOT) if DRIVE_ROOT.exists() else "empty")

## 4. Cấu hình

Sửa paths cho khớp với layout Drive của ông. Mặc định: tất cả files để trong `/content/drive/MyDrive/mRAG/`.

In [ ]:
import glob
from dataclasses import dataclass, field
from typing import Optional


def _auto_find_basket_pickle(drive_root: Path) -> str:
    candidates = sorted(glob.glob(str(drive_root / "eval_baskets*.pkl")))
    if not candidates:
        candidates = sorted(glob.glob(str(drive_root / "*.pkl")))
    if not candidates:
        raise FileNotFoundError(f"Không tìm thấy file pkl nào trong {drive_root}. Upload eval_baskets.pkl lên Drive trước.")
    return candidates[0]


@dataclass
class Item2VecConfig:
    basket_cache_pickle: str = field(default_factory=lambda: _auto_find_basket_pickle(DRIVE_ROOT))
    meta_csv: str = str(DRIVE_ROOT / "dataset_qwen_completed.csv")
    output_dir: str = str(DRIVE_ROOT / "item2vec_output")

    embed_dim: int = 256
    num_negatives: int = 10
    hard_negative_ratio: float = 0.6
    min_item_count: int = 5
    filter_same_pt_positives: bool = True

    epochs: int = 10
    batch_size: int = 32768
    lr: float = 5e-3
    weight_decay: float = 0.0
    use_bf16: bool = True

    seed: int = 42
    knn_k: int = 20
    knn_chunk: int = 4096


config = Item2VecConfig()
Path(config.output_dir).mkdir(parents=True, exist_ok=True)

print("Files trong Drive root:", os.listdir(DRIVE_ROOT))
print()
print(f"basket_cache_pickle: {config.basket_cache_pickle}")
print(f"  -> {'OK' if os.path.exists(config.basket_cache_pickle) else 'MISSING'}")
print(f"meta_csv: {config.meta_csv}")
print(f"  -> {'OK' if os.path.exists(config.meta_csv) else 'MISSING'}")
print()
print(f"filter_same_pt_positives = {config.filter_same_pt_positives}")
print("  (True = chỉ giữ cross-PT pairs lúc training, fix gradient cancellation từ v2A.0)")

assert os.path.exists(config.basket_cache_pickle), "Thiếu basket pkl"
assert os.path.exists(config.meta_csv), "Thiếu meta CSV"

## 5. Loader dữ liệu

Load `train_baskets` từ pkl. Tin tưởng pkl đã được build đúng ở local (`OutfitGraphEvaluator.prepare()`).

In [ ]:
import pickle
import logging
import time
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s | %(message)s")
log = logging.getLogger("item2vec")


class BasketLoader:
    def __init__(self, config: Item2VecConfig):
        self.config = config

    def load(self) -> List[List[str]]:
        with open(self.config.basket_cache_pickle, "rb") as handle:
            payload = pickle.load(handle)
        baskets = payload["train_baskets"]
        log.info("loaded %d train baskets from cache", len(baskets))
        return baskets


loader = BasketLoader(config)
baskets = loader.load()
print(f"basket count: {len(baskets):,}")
print(f"basket size sample (first 5): {[len(b) for b in baskets[:5]]}")

## 6. Vocab + product_type index + pair materialization

**Thứ tự**: Vocab → ProductTypeIndex → PairMaterializer (pair builder cần PT lookup nếu `filter_same_pt_positives=True`).

- **Vocab**: chỉ giữ items với count >= `min_item_count`.
- **ProductTypeIndex**: map vocab idx → PT idx + cấu trúc CSR-like cho hard negative sampling trên GPU.
- **PairMaterializer**: mỗi basket size n → n*(n-1) directed pairs. Nếu `filter_same_pt_positives=True`, lọc bỏ pairs cùng PT (pairs trong cùng category trong giỏ hàng, vd. 2 áo) — fix gradient cancellation cho cross-PT eval.

In [ ]:
from collections import Counter


class VocabBuilder:
    def __init__(self, config: Item2VecConfig):
        self.config = config
        self.vocab: Dict[str, int] = {}
        self.id_to_item: List[str] = []
        self.item_counts: np.ndarray = np.empty(0, dtype=np.int64)

    def build(self, baskets: List[List[str]]) -> None:
        counter: Counter = Counter()
        for basket in tqdm(baskets, desc="counting items"):
            for item in basket:
                counter[item] += 1
        kept = sorted(item for item, count in counter.items() if count >= self.config.min_item_count)
        self.vocab = {item: idx for idx, item in enumerate(kept)}
        self.id_to_item = kept
        self.item_counts = np.array([counter[item] for item in kept], dtype=np.int64)
        log.info("vocab size: %d", len(self.vocab))


class ProductTypeIndex:
    def __init__(self, config: Item2VecConfig, vocab: Dict[str, int]):
        self.config = config
        self.vocab = vocab
        self.pt_for_item: np.ndarray = np.empty(0, dtype=np.int32)
        self.pt_offsets: np.ndarray = np.empty(0, dtype=np.int64)
        self.pt_sizes: np.ndarray = np.empty(0, dtype=np.int64)
        self.items_by_pt: np.ndarray = np.empty(0, dtype=np.int32)
        self.num_pts: int = 0

    def build(self) -> None:
        df = pd.read_csv(self.config.meta_csv, usecols=["article_id", "product_type_name"], dtype=str)
        df["article_id"] = df["article_id"].str.zfill(10)
        df["product_type_name"] = df["product_type_name"].fillna("")
        pt_map = dict(zip(df["article_id"], df["product_type_name"]))

        pt_to_idx: Dict[str, int] = {}
        vocab_size = len(self.vocab)
        pt_for_item = np.full(vocab_size, -1, dtype=np.int32)
        for item, idx in self.vocab.items():
            pt = pt_map.get(item, "")
            if not pt:
                continue
            if pt not in pt_to_idx:
                pt_to_idx[pt] = len(pt_to_idx)
            pt_for_item[idx] = pt_to_idx[pt]
        self.pt_for_item = pt_for_item
        self.num_pts = len(pt_to_idx)

        items_by_pt: List[List[int]] = [[] for _ in range(self.num_pts)]
        for idx, pt_idx in enumerate(pt_for_item.tolist()):
            if pt_idx >= 0:
                items_by_pt[pt_idx].append(idx)
        flat_items: List[int] = []
        pt_offsets = np.zeros(self.num_pts, dtype=np.int64)
        pt_sizes = np.zeros(self.num_pts, dtype=np.int64)
        for pt_idx, items in enumerate(items_by_pt):
            pt_offsets[pt_idx] = len(flat_items)
            pt_sizes[pt_idx] = len(items)
            flat_items.extend(items)
        self.items_by_pt = np.asarray(flat_items, dtype=np.int32)
        self.pt_offsets = pt_offsets
        self.pt_sizes = pt_sizes
        log.info("indexed %d product_types", self.num_pts)


class PairMaterializer:
    def __init__(self, vocab: Dict[str, int], pt_for_item: Optional[np.ndarray] = None, filter_same_pt: bool = False):
        self.vocab = vocab
        self.pt_for_item = pt_for_item
        self.filter_same_pt = filter_same_pt and pt_for_item is not None
        self.pair_anchors: np.ndarray = np.empty(0, dtype=np.int32)
        self.pair_positives: np.ndarray = np.empty(0, dtype=np.int32)
        self.dropped_same_pt: int = 0
        self.kept_pairs: int = 0

    def build(self, baskets: List[List[str]]) -> None:
        anchor_chunks: List[np.ndarray] = []
        positive_chunks: List[np.ndarray] = []
        total_off_diag = 0
        total_kept = 0
        for basket in tqdm(baskets, desc="materializing pairs"):
            mapped = np.asarray([self.vocab[item] for item in basket if item in self.vocab], dtype=np.int32)
            n = mapped.size
            if n < 2:
                continue
            grid_a, grid_p = np.meshgrid(mapped, mapped, indexing="ij")
            off_diag = ~np.eye(n, dtype=bool)
            total_off_diag += int(off_diag.sum())
            if self.filter_same_pt:
                pt_a = self.pt_for_item[grid_a]
                pt_p = self.pt_for_item[grid_p]
                cross_pt = (pt_a != pt_p) | (pt_a < 0) | (pt_p < 0)
                final_mask = off_diag & cross_pt
            else:
                final_mask = off_diag
            total_kept += int(final_mask.sum())
            anchor_chunks.append(grid_a[final_mask])
            positive_chunks.append(grid_p[final_mask])
        self.pair_anchors = np.concatenate(anchor_chunks).astype(np.int32, copy=False)
        self.pair_positives = np.concatenate(positive_chunks).astype(np.int32, copy=False)
        self.dropped_same_pt = total_off_diag - total_kept
        self.kept_pairs = total_kept
        if self.filter_same_pt:
            drop_pct = 100.0 * self.dropped_same_pt / max(1, total_off_diag)
            log.info("materialized %d pairs (dropped %d same-PT pairs = %.1f%% of off-diag)",
                     self.pair_anchors.size, self.dropped_same_pt, drop_pct)
        else:
            log.info("materialized %d pairs (no PT filter)", self.pair_anchors.size)


vocab_builder = VocabBuilder(config)
vocab_builder.build(baskets)

pt_index = ProductTypeIndex(config, vocab_builder.vocab)
pt_index.build()

pair_mat = PairMaterializer(
    vocab_builder.vocab,
    pt_for_item=pt_index.pt_for_item,
    filter_same_pt=config.filter_same_pt_positives,
)
pair_mat.build(baskets)

del baskets
import gc; gc.collect()
print(f"vocab size: {len(vocab_builder.vocab):,}")
print(f"product types: {pt_index.num_pts}")
print(f"pairs (kept): {pair_mat.pair_anchors.size:,}")
if config.filter_same_pt_positives:
    print(f"pairs dropped (same-PT): {pair_mat.dropped_same_pt:,}")

## 7. Model definition

Standard SGNS:
- 2 embedding tables: `in_emb` cho anchor, `out_emb` cho context (positive + negative).
- Init: in_emb ~ N(0, 1/sqrt(d)), out_emb = 0 (standard word2vec init).
- Loss: `-log σ(score(a, pos)) - mean_k log σ(-score(a, neg_k))`.
- Inference: dùng `in_emb` đã L2-normalize → cosine kNN.

In [ ]:
import math
import torch.nn as nn
import torch.nn.functional as F


class Item2VecModel(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int):
        super().__init__()
        self.in_emb = nn.Embedding(vocab_size, embed_dim)
        self.out_emb = nn.Embedding(vocab_size, embed_dim)
        nn.init.normal_(self.in_emb.weight, std=1.0 / math.sqrt(embed_dim))
        nn.init.zeros_(self.out_emb.weight)

    def positive_score(self, anchors: torch.Tensor, positives: torch.Tensor) -> torch.Tensor:
        a = self.in_emb(anchors)
        p = self.out_emb(positives)
        return (a * p).sum(dim=-1)

    def negative_scores(self, anchors: torch.Tensor, negatives: torch.Tensor) -> torch.Tensor:
        a = self.in_emb(anchors).unsqueeze(1)
        n = self.out_emb(negatives)
        return (a * n).sum(dim=-1)

    def normalized_input_embedding(self) -> torch.Tensor:
        with torch.no_grad():
            return F.normalize(self.in_emb.weight.detach().float(), p=2, dim=-1)


def sgns_loss(pos_scores: torch.Tensor, neg_scores: torch.Tensor) -> torch.Tensor:
    pos_loss = -F.logsigmoid(pos_scores).mean()
    neg_loss = -F.logsigmoid(-neg_scores).mean()
    return pos_loss + neg_loss

## 8. Trainer — GPU-native negative sampling + bf16 autocast

Khác bản local CPU:
- Tất cả sampling negatives chạy trên GPU (`torch.rand`, `torch.multinomial`) → không có CPU-GPU transfer mỗi batch.
- Mixed precision bf16 thông qua `torch.autocast` → matmul nhanh ~2x trên A100, không cần GradScaler vì bf16 dynamic range tương đương fp32.
- Master weights vẫn fp32 trong Adam state.

In [ ]:
import json


class Item2VecTrainer:
    def __init__(self, config: Item2VecConfig, vocab_builder: VocabBuilder, pair_mat: PairMaterializer, pt_index: ProductTypeIndex):
        self.config = config
        self.vocab_builder = vocab_builder
        self.pair_mat = pair_mat
        self.pt_index = pt_index
        self.device = torch.device("cuda")
        self.model = Item2VecModel(len(vocab_builder.vocab), config.embed_dim).to(self.device)
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
        self.loss_history: List[float] = []

        torch.manual_seed(config.seed)

        self._pair_anchors_gpu = torch.from_numpy(pair_mat.pair_anchors).long().to(self.device)
        self._pair_positives_gpu = torch.from_numpy(pair_mat.pair_positives).long().to(self.device)
        self._pt_for_item = torch.from_numpy(pt_index.pt_for_item).long().to(self.device)
        self._pt_offsets = torch.from_numpy(pt_index.pt_offsets).long().to(self.device)
        self._pt_sizes = torch.from_numpy(pt_index.pt_sizes).long().to(self.device)
        self._items_by_pt = torch.from_numpy(pt_index.items_by_pt).long().to(self.device)

        counts = vocab_builder.item_counts.astype(np.float64) ** 0.75
        probs = counts / counts.sum()
        self._unigram_dist = torch.from_numpy(probs).float().to(self.device)

        log.info("trainer initialized on %s | model params: %d", self.device, sum(p.numel() for p in self.model.parameters()))

    def _sample_negatives(self, anchor_ids: torch.Tensor) -> torch.Tensor:
        B = anchor_ids.size(0)
        N = self.config.num_negatives
        n_hard = int(round(N * self.config.hard_negative_ratio))
        n_soft = N - n_hard
        vocab_size = len(self.vocab_builder.vocab)

        anchor_pts = self._pt_for_item[anchor_ids]
        safe_pts = anchor_pts.clamp(min=0)
        sizes = self._pt_sizes[safe_pts]
        offsets = self._pt_offsets[safe_pts]
        no_pt_mask = (anchor_pts < 0) | (sizes == 0)

        out = torch.empty((B, N), dtype=torch.long, device=self.device)
        if n_hard > 0:
            sizes_safe = sizes.clamp(min=1)
            rand = torch.rand((B, n_hard), device=self.device)
            hard_rand = (rand * sizes_safe.unsqueeze(1).float()).long()
            flat_idx = offsets.unsqueeze(1) + hard_rand
            hard_samples = self._items_by_pt[flat_idx]
            fallback = torch.randint(0, vocab_size, (B, n_hard), device=self.device)
            mask = no_pt_mask.unsqueeze(1).expand(-1, n_hard)
            out[:, :n_hard] = torch.where(mask, fallback, hard_samples)
        if n_soft > 0:
            soft = torch.multinomial(self._unigram_dist, B * n_soft, replacement=True).view(B, n_soft)
            out[:, n_hard:] = soft
        return out

    def train_epoch(self, epoch: int) -> float:
        bs = self.config.batch_size
        num_pairs = self._pair_anchors_gpu.size(0)
        gen = torch.Generator(device=self.device)
        gen.manual_seed(self.config.seed + epoch)
        order = torch.randperm(num_pairs, device=self.device, generator=gen)

        self.model.train()
        total_loss = 0.0
        n_batches = 0
        amp_dtype = torch.bfloat16 if self.config.use_bf16 else torch.float32

        num_batches = (num_pairs + bs - 1) // bs
        pbar = tqdm(range(0, num_pairs, bs), total=num_batches, desc=f"epoch {epoch+1}/{self.config.epochs}", leave=True)
        for start in pbar:
            idx = order[start : start + bs]
            anchors = self._pair_anchors_gpu[idx]
            positives = self._pair_positives_gpu[idx]
            negatives = self._sample_negatives(anchors)

            with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=self.config.use_bf16):
                pos_scores = self.model.positive_score(anchors, positives)
                neg_scores = self.model.negative_scores(anchors, negatives)
                loss = sgns_loss(pos_scores, neg_scores)

            self.optimizer.zero_grad(set_to_none=True)
            loss.backward()
            self.optimizer.step()

            total_loss += loss.item()
            n_batches += 1
            if n_batches % 50 == 0:
                pbar.set_postfix(loss=f"{total_loss / n_batches:.4f}")

        return total_loss / max(1, n_batches)

    def train(self) -> None:
        for epoch in range(self.config.epochs):
            start = time.time()
            avg_loss = self.train_epoch(epoch)
            self.loss_history.append(avg_loss)
            log.info("epoch %d done in %.1fs avg_loss=%.4f", epoch + 1, time.time() - start, avg_loss)

    def build_knn_graph(self) -> List[Tuple[str, str, float]]:
        emb = self.model.normalized_input_embedding()
        vocab_size = emb.size(0)
        K = self.config.knn_k
        chunk = self.config.knn_chunk
        edges: List[Tuple[str, str, float]] = []
        for start in tqdm(range(0, vocab_size, chunk), desc="building kNN graph"):
            end = min(start + chunk, vocab_size)
            chunk_emb = emb[start:end]
            sims = chunk_emb @ emb.t()
            diag_idx = torch.arange(end - start, device=self.device)
            sims[diag_idx, start + diag_idx] = -2.0
            top_sims, top_idx = sims.topk(K, dim=-1)
            top_sims_cpu = top_sims.cpu().numpy()
            top_idx_cpu = top_idx.cpu().numpy()
            for local_idx in range(end - start):
                anchor_id = self.vocab_builder.id_to_item[start + local_idx]
                for rank in range(K):
                    nbr_idx = int(top_idx_cpu[local_idx, rank])
                    sim = float(top_sims_cpu[local_idx, rank])
                    edges.append((anchor_id, self.vocab_builder.id_to_item[nbr_idx], sim))
        return edges

    def export(self, output_dir: str) -> Dict[str, str]:
        tag = f"d{self.config.embed_dim}_n{self.config.num_negatives}_h{int(self.config.hard_negative_ratio * 100)}_e{self.config.epochs}"
        out_path = Path(output_dir)
        out_path.mkdir(parents=True, exist_ok=True)

        edges = self.build_knn_graph()
        df = pd.DataFrame(edges, columns=["item_a", "item_b", "weight"])
        df["weight"] = (df["weight"].clip(lower=0.0) * 1000.0).astype(int)
        df = df.sort_values(by=["item_a", "weight"], ascending=[True, False])
        knn_path = out_path / f"item2vec_{tag}_k{self.config.knn_k}.csv"
        df.to_csv(knn_path, index=False)
        log.info("wrote knn graph: %s rows=%d", knn_path, len(df))

        emb = self.model.normalized_input_embedding().cpu().numpy()
        emb_path = out_path / f"item2vec_{tag}_embeddings.npz"
        np.savez_compressed(emb_path, embeddings=emb, items=np.array(self.vocab_builder.id_to_item))
        log.info("saved embeddings: %s shape=%s", emb_path, emb.shape)

        meta_path = out_path / f"item2vec_{tag}_metadata.json"
        meta = {
            "config": {k: v for k, v in self.config.__dict__.items()},
            "vocab_size": len(self.vocab_builder.vocab),
            "num_pairs": int(self.pair_mat.pair_anchors.size),
            "loss_history": self.loss_history,
            "knn_edges": len(df),
        }
        with open(meta_path, "w") as f:
            json.dump(meta, f, indent=2)
        log.info("wrote metadata: %s", meta_path)

        return {"knn": str(knn_path), "embeddings": str(emb_path), "metadata": str(meta_path)}

## 9. Train

Bấm chạy. Trên A100 dự kiến 5-10 phút.

In [ ]:
trainer = Item2VecTrainer(config, vocab_builder, pair_mat, pt_index)
trainer.train()

## 10. Export artifacts về Drive

Output files được lưu vào `config.output_dir` (mặc định: `/content/drive/MyDrive/mRAG/item2vec_output/`). Tải về local + cho vào `data/processed/graph_experiments/` để eval qua harness.

In [ ]:
artifacts = trainer.export(config.output_dir)
for k, v in artifacts.items():
    size_mb = os.path.getsize(v) / 1024**2
    print(f"{k}: {v} ({size_mb:.1f} MB)")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(trainer.loss_history) + 1), trainer.loss_history, marker="o")
plt.xlabel("epoch")
plt.ylabel("avg SGNS loss")
plt.title("Training loss curve")
plt.grid(True, alpha=0.3)
plt.show()

## 11. Eval trực tiếp trong notebook

Port logic từ `src/scripts/eval_graph.py` của repo local. Cùng dataset (test_baskets từ pkl), cùng metric (recall@k, MAP@k, hit@k), cùng dual ground-truth mode (`all` / `cross_product_type`).

Baseline để so sánh: **cobuy_clean_legacy cross-PT recall@10 = 0.0542**. Đây là target Phase 2A cần vượt.

In [ ]:
from typing import Sequence


class InNotebookEvaluator:
    def __init__(self, basket_cache_pickle: str, meta_csv: str, max_test_baskets: int = 30000, seed: int = 42):
        with open(basket_cache_pickle, "rb") as handle:
            payload = pickle.load(handle)
        test_baskets = payload["test_baskets"]
        if len(test_baskets) > max_test_baskets:
            import random
            rng = random.Random(seed)
            rng.shuffle(test_baskets)
            test_baskets = test_baskets[:max_test_baskets]
        self.test_baskets = test_baskets
        log.info("eval test baskets: %d", len(test_baskets))

        df = pd.read_csv(meta_csv, usecols=["article_id", "product_type_name"], dtype=str)
        df["article_id"] = df["article_id"].str.zfill(10)
        self.pt_lookup = dict(zip(df["article_id"], df["product_type_name"].fillna("")))

    @staticmethod
    def _accumulate(ranked: Sequence[str], gt: set, k_values: Sequence[int], recall_acc, map_acc, hit_acc):
        gt_size = len(gt)
        relevant = 0
        precision_sum = 0.0
        for rank, nid in enumerate(ranked, start=1):
            if nid in gt:
                relevant += 1
                precision_sum += relevant / rank
            for k in k_values:
                if rank == k:
                    recall_acc[k] += relevant / gt_size
                    map_acc[k] += precision_sum / min(k, gt_size)
                    if relevant > 0:
                        hit_acc[k] += 1
        max_rank = len(ranked)
        for k in k_values:
            if max_rank >= k:
                continue
            recall_acc[k] += relevant / gt_size
            map_acc[k] += precision_sum / min(k, gt_size)
            if relevant > 0:
                hit_acc[k] += 1

    def evaluate(self, adjacency: Dict[str, List[Tuple[str, float]]], k_values=(5, 10, 20), gt_mode: str = "all") -> Dict:
        max_k = max(k_values)
        recall_acc = {k: 0.0 for k in k_values}
        map_acc = {k: 0.0 for k in k_values}
        hit_acc = {k: 0 for k in k_values}
        anchors_with_neighbors = 0
        anchors_total = 0

        for basket in tqdm(self.test_baskets, desc=f"eval {gt_mode}"):
            basket_set = set(basket)
            for anchor in basket:
                gt = basket_set - {anchor}
                if gt_mode == "cross_product_type":
                    anchor_pt = self.pt_lookup.get(anchor, "")
                    if not anchor_pt:
                        continue
                    gt = {i for i in gt if self.pt_lookup.get(i, "") != anchor_pt}
                if not gt:
                    continue
                anchors_total += 1
                neighbors = adjacency.get(anchor)
                if not neighbors:
                    continue
                anchors_with_neighbors += 1
                top_neighbors = [nid for nid, _ in neighbors[:max_k]]
                self._accumulate(top_neighbors, gt, k_values, recall_acc, map_acc, hit_acc)

        denom = max(1, anchors_with_neighbors)
        return {
            "mode": gt_mode,
            "coverage": anchors_with_neighbors / max(1, anchors_total),
            "recall_at_k": {k: recall_acc[k] / denom for k in k_values},
            "map_at_k": {k: map_acc[k] / denom for k in k_values},
            "hit_at_k": {k: hit_acc[k] / denom for k in k_values},
            "anchors_with_neighbors": anchors_with_neighbors,
            "anchors_total": anchors_total,
        }


def build_adjacency_from_csv(csv_path: str) -> Dict[str, List[Tuple[str, float]]]:
    df = pd.read_csv(csv_path, dtype={"item_a": str, "item_b": str})
    df["item_a"] = df["item_a"].str.zfill(10)
    df["item_b"] = df["item_b"].str.zfill(10)
    adj: Dict[str, List[Tuple[str, float]]] = {}
    for row in df.itertuples(index=False):
        adj.setdefault(row.item_a, []).append((row.item_b, float(row.weight)))
    for a in adj:
        adj[a].sort(key=lambda x: x[1], reverse=True)
    return adj


adjacency = build_adjacency_from_csv(artifacts["knn"])
log.info("adjacency loaded: %d anchors", len(adjacency))

evaluator = InNotebookEvaluator(config.basket_cache_pickle, config.meta_csv)
result_all = evaluator.evaluate(adjacency, gt_mode="all")
result_cross = evaluator.evaluate(adjacency, gt_mode="cross_product_type")


def print_result(result: Dict) -> None:
    print(f"mode={result['mode']}")
    print(f"  coverage:  {result['coverage']:.4f}  ({result['anchors_with_neighbors']:,} / {result['anchors_total']:,} anchors)")
    for k in sorted(result["recall_at_k"].keys()):
        print(f"  recall@{k:<2} = {result['recall_at_k'][k]:.4f}   map@{k:<2} = {result['map_at_k'][k]:.4f}   hit@{k:<2} = {result['hit_at_k'][k]:.4f}")


print_result(result_all)
print()
print_result(result_cross)
print()
baseline_cross_recall10 = 0.0542
item2vec_cross_recall10 = result_cross["recall_at_k"][10]
delta = item2vec_cross_recall10 - baseline_cross_recall10
verdict = "WIN" if delta > 0 else "LOSS"
print(f"baseline cobuy_clean_legacy cross-PT recall@10: {baseline_cross_recall10:.4f}")
print(f"item2vec cross-PT recall@10:                    {item2vec_cross_recall10:.4f}")
print(f"delta: {delta:+.4f}  -> {verdict}")

## 12. Bước tiếp theo

### Variant đang chạy: V2A.1 (filter same-PT positives)

**V2A.0** (filter=False) đã cho cross-PT recall@10 = 0.0317 (LOSS so với baseline 0.0542). Nguyên nhân: gradient cancellation giữa positive same-PT pairs và hard negative same-PT samples → model học similarity thay vì pairing.

**V2A.1** (filter=True) lọc same-PT pairs khỏi training data — tương tự rule_2 của cobuy_clean_legacy. Kỳ vọng cross-PT recall tăng đáng kể.

### Sau khi chạy xong

- **Nếu cross-PT recall@10 > 0.0542 (WIN)**: chốt V2A.1, tải về `item2vec_{tag}_k20.csv` + metadata.json + embeddings.npz, gửi tôi để viết step 2A report.
- **Nếu vẫn LOSS hoặc marginal**: tune tiếp theo thứ tự (đổi 1 knob mỗi lần, re-run từ cell-12 hoặc cell-18 tuỳ knob):
  1. **Tăng `hard_negative_ratio` (0.6 → 0.8)**: ép cross-PT signal mạnh hơn (giờ không bị cancel với positives nữa).
  2. **Tăng `epochs` (10 → 20)**: data ít hơn sau filter → cần nhiều epochs hơn để hội tụ.
  3. **Tăng `num_negatives` (10 → 15-20)**: nhiều contrast hơn.
  4. **Tăng `embed_dim` (256 → 384)**: nhiều capacity hơn.
  5. **Subsampling popular items** (Mikolov 2013): pre-filter pairs có anchor là top-frequency items với probability `1 - sqrt(t/f)`, t=1e-4. Giảm Trousers/T-shirt dominance.

Mỗi config: ghi lại vào sheet để track. Đừng tune nhiều knob cùng lúc — không biết knob nào tạo signal.

### Local validation (optional)

```powershell
conda activate mRAG
$env:GRAPH_FILE = "data/processed/graph_experiments/item2vec_{tag}_k20.csv"
python -m src.scripts.eval_graph
```
Số phải khớp với in-notebook eval (cùng test_baskets, cùng metric).